# Download PRISM Climate Data for Study Area

In this notebook, the study area boundary created in Notebook 01 will be used to download and save PRISM climate data for future use. 

**Add some info on the PRISM dataset and links**


## Step 1: Import Libraries and set Project Directories

In [1]:
# import libraries

# File management
import os
import pathlib
from pathlib import Path
import sys # for importing src files
#import pyarrow # saving GeoParquet files

# Downloading
from tqdm.notebook import tqdm # progress bar
import requests # for SNOTEL API access
import zipfile

# Data Management
import pandas as pd
import xarray as xr

# Geospatial Management
import geopandas as gpd
from pygeohydro import WBD # site boundary based on watersheds
#import rasterio
#import rioxarray as rxr
#from shapely.geometry import box # lat/lon boundary boxes
#from shapely.ops import split
#from shapely.ops import unary_union # for intersecting boundaries

# Plotting
import matplotlib
import matplotlib.pyplot as plt
import holoviews as hv
import hvplot.pandas

print('Imports complete!')


Imports complete!


In [2]:
# Force Jupyter to reload external modules automatically before executing a cell
# I used Gemini to help me set up the connection to my src files

# force reimporting of src scripts
%load_ext autoreload
%autoreload 2

# Find the parent directory of this notebook (the repo root) and add it to Python's search path
repo_root = str(Path.cwd().parent)
if repo_root not in sys.path:
    sys.path.append(repo_root)

# Now Python can see the src folder
# import just specific functions
from src import prism

# once I'm using environment.yml I can get rid of the above steps

# from src
from src import prism

print('src import complete!')

src import complete!


In [3]:
# reload src scripts
%reload_ext autoreload

In [4]:
# set directories
proj_dir = os.path.join(pathlib.Path.home(),
                        'Documents',
                        'Graduate_School',
                        'EDA_Certificate', 
                        'Summer', 
                        'snow-drought-modeling')
os.makedirs(proj_dir, exist_ok=True)

raw_data_dir = os.path.join(proj_dir, 'data', 'raw')
os.makedirs(raw_data_dir, exist_ok=True)

cleaned_data_dir = os.path.join(proj_dir, 'data', 'cleaned')
os.makedirs(raw_data_dir, exist_ok=True)

## Step 2: Calculate Water Years

This step will calculate daily dates for the range of water years required.

In [5]:
# calculate water year dates

wy_dates = prism.water_year_dates(start_wy=1991, end_wy=2020)
wy_dates

[datetime.datetime(1990, 10, 1, 0, 0),
 datetime.datetime(1990, 10, 2, 0, 0),
 datetime.datetime(1990, 10, 3, 0, 0),
 datetime.datetime(1990, 10, 4, 0, 0),
 datetime.datetime(1990, 10, 5, 0, 0),
 datetime.datetime(1990, 10, 6, 0, 0),
 datetime.datetime(1990, 10, 7, 0, 0),
 datetime.datetime(1990, 10, 8, 0, 0),
 datetime.datetime(1990, 10, 9, 0, 0),
 datetime.datetime(1990, 10, 10, 0, 0),
 datetime.datetime(1990, 10, 11, 0, 0),
 datetime.datetime(1990, 10, 12, 0, 0),
 datetime.datetime(1990, 10, 13, 0, 0),
 datetime.datetime(1990, 10, 14, 0, 0),
 datetime.datetime(1990, 10, 15, 0, 0),
 datetime.datetime(1990, 10, 16, 0, 0),
 datetime.datetime(1990, 10, 17, 0, 0),
 datetime.datetime(1990, 10, 18, 0, 0),
 datetime.datetime(1990, 10, 19, 0, 0),
 datetime.datetime(1990, 10, 20, 0, 0),
 datetime.datetime(1990, 10, 21, 0, 0),
 datetime.datetime(1990, 10, 22, 0, 0),
 datetime.datetime(1990, 10, 23, 0, 0),
 datetime.datetime(1990, 10, 24, 0, 0),
 datetime.datetime(1990, 10, 25, 0, 0),
 datetime

In [ ]:
test_dates = [d for d in wy_dates if 1991 <= d.year <= 1992 and d.month < 7]
test_dates

[datetime.datetime(1991, 1, 1, 0, 0),
 datetime.datetime(1991, 1, 2, 0, 0),
 datetime.datetime(1991, 1, 3, 0, 0),
 datetime.datetime(1991, 1, 4, 0, 0),
 datetime.datetime(1991, 1, 5, 0, 0),
 datetime.datetime(1991, 1, 6, 0, 0),
 datetime.datetime(1991, 1, 7, 0, 0),
 datetime.datetime(1991, 1, 8, 0, 0),
 datetime.datetime(1991, 1, 9, 0, 0),
 datetime.datetime(1991, 1, 10, 0, 0),
 datetime.datetime(1991, 1, 11, 0, 0),
 datetime.datetime(1991, 1, 12, 0, 0),
 datetime.datetime(1991, 1, 13, 0, 0),
 datetime.datetime(1991, 1, 14, 0, 0),
 datetime.datetime(1991, 1, 15, 0, 0),
 datetime.datetime(1991, 1, 16, 0, 0),
 datetime.datetime(1991, 1, 17, 0, 0),
 datetime.datetime(1991, 1, 18, 0, 0),
 datetime.datetime(1991, 1, 19, 0, 0),
 datetime.datetime(1991, 1, 20, 0, 0),
 datetime.datetime(1991, 1, 21, 0, 0),
 datetime.datetime(1991, 1, 22, 0, 0),
 datetime.datetime(1991, 1, 23, 0, 0),
 datetime.datetime(1991, 1, 24, 0, 0),
 datetime.datetime(1991, 1, 25, 0, 0),
 datetime.datetime(1991, 1, 26, 0,

In [18]:
from datetime import datetime


In [22]:

end_year = 2020
start_year = end_year -1

end_year
start_year


2019

In [23]:
start_range = datetime(start_year, 10, 1)
end_range = datetime(end_year, 6, 1)

start_range
end_range

datetime.datetime(2020, 6, 1, 0, 0)

In [24]:

test_dates = [dt for dt in wy_dates if start_range <= dt <= end_range]
test_dates

[datetime.datetime(2019, 10, 1, 0, 0),
 datetime.datetime(2019, 10, 2, 0, 0),
 datetime.datetime(2019, 10, 3, 0, 0),
 datetime.datetime(2019, 10, 4, 0, 0),
 datetime.datetime(2019, 10, 5, 0, 0),
 datetime.datetime(2019, 10, 6, 0, 0),
 datetime.datetime(2019, 10, 7, 0, 0),
 datetime.datetime(2019, 10, 8, 0, 0),
 datetime.datetime(2019, 10, 9, 0, 0),
 datetime.datetime(2019, 10, 10, 0, 0),
 datetime.datetime(2019, 10, 11, 0, 0),
 datetime.datetime(2019, 10, 12, 0, 0),
 datetime.datetime(2019, 10, 13, 0, 0),
 datetime.datetime(2019, 10, 14, 0, 0),
 datetime.datetime(2019, 10, 15, 0, 0),
 datetime.datetime(2019, 10, 16, 0, 0),
 datetime.datetime(2019, 10, 17, 0, 0),
 datetime.datetime(2019, 10, 18, 0, 0),
 datetime.datetime(2019, 10, 19, 0, 0),
 datetime.datetime(2019, 10, 20, 0, 0),
 datetime.datetime(2019, 10, 21, 0, 0),
 datetime.datetime(2019, 10, 22, 0, 0),
 datetime.datetime(2019, 10, 23, 0, 0),
 datetime.datetime(2019, 10, 24, 0, 0),
 datetime.datetime(2019, 10, 25, 0, 0),
 datetime

## Step 3: Download PRISM rasters for ML training and application

In [26]:
# set dirs here
dnld_dir = os.path.join(raw_data_dir, 'prism', 'full_rasters')
os.makedirs(dnld_dir, exist_ok=True)

In [25]:
from datetime import datetime
test_dates = [datetime(1987, 11, 1)]
test_dates

[datetime.datetime(1987, 11, 1, 0, 0)]

In [26]:
# test download func
test_files = prism.get_prism_snow_rasters(test_dates, '4km', dnld_dir, variables)

Processing PRISM Rasters: 100%|██████████| 3/3 [00:12<00:00,  4.10s/file]

Download complete.


download func took 9.4 sec for one date, three variables when downloading in GeoTIFF format. 
netCDF format instead: 11.1 sec
netCDF, deleting files: 13.2 sec

probably still worth using the NetCDF format 

In [ ]:
# try to download netCDF rasters in yearly chunks

# define year range and variables
# range has to be one year more than actual range desired - investigate
all_years = range(1991, 2021)
variables = ['ppt', 'tmin', 'tmax']

# loop through each year, printing an update when each yearly batch is done. 
# This download will take some time, so this method helps monitor progress.
for year in all_years:
    print(f"--- Starting Batch for Water Year {year} ---")
    
    # Get dates for this specific winter
    end_year = year
    start_year = year -1
    start_range = datetime(start_year, 10, 1)
    end_range = datetime(end_year, 6, 1)
    year_dates = [dt for dt in wy_dates if start_range <= dt <= end_range]
    
    # Run the function for just this year
    prism.get_prism_rasters(year_dates, '4km', dnld_dir, variables)
    
    # Print an update
    print(f"--- Finished Batch {year}. Moving to next. ---")

--- Starting Batch for Water Year 2020 ---


Processing PRISM Rasters: 100%|██████████| 735/735 [43:15<00:00,  3.53s/file]

Download complete.
--- Finished Batch 2020. Moving to next. ---


## Step 4: Crop Rasters to Study Boundary

The rasters acquired from PRISM are for the entire continental US; that's obviously much larger than a HUC6 watershed, and will be too unwieldy to work with. This step will crop the rasters to the study boundary for easier use.

In [9]:
# set paths
mhw_path = Path(raw_data_dir, 'boundaries', 'mhw_gdf.parquet')
mhw_train_path = Path(cleaned_data_dir, 'boundaries', 'mhw_buff_clip_gdf.parquet')
prism_rast_path = Path(raw_data_dir, 'prism', 'full_rasters')

In [6]:
# load the mhw boundary gdf
mhw_gdf = gpd.read_parquet(mhw_path)

# and the buffered gdf
mhw_train_gdf = gpd.read_parquet(mhw_train_path)

In [10]:
prism_rast_path

WindowsPath('C:/Users/Raini/Documents/Graduate_School/EDA_Certificate/Summer/snow-drought-modeling/data/raw/prism/full_rasters')

In [11]:
# use crop_prism_rasters from src to crop using the mhw_gdf
prism_mhw_train_raw = prism.crop_prism_rasters(prism_rast_path, mhw_train_gdf)

ValueError: Could not find any dimension coordinates to use to order the Dataset objects for concatenation

In [33]:
test_nc = xr.open_dataset(f'{dnld_dir}/prism_ppt_us_25m_19901001.nc')
test_nc

<xarray.Dataset> Size: 4MB
Dimensions:  (lat: 621, lon: 1405)
Coordinates:
  * lat      (lat) float64 5kB 24.08 24.12 24.17 24.21 ... 49.83 49.88 49.92
  * lon      (lon) float64 11kB -125.0 -125.0 -124.9 ... -66.58 -66.54 -66.5
Data variables:
    crs      |S1 1B ...
    Band1    (lat, lon) float32 3MB ...
Attributes:
    Conventions:  CF-1.5
    GDAL:         GDAL 3.12.0 "Chicoutimi", released 2025/11/03
    history:      Thu Jun 18 17:05:52 2026: GDAL CreateCopy( /nfs/pancake/u5/...

In [34]:
test_nc.lat

<xarray.DataArray 'lat' (lat: 621)> Size: 5kB
array([24.083333, 24.125   , 24.166667, ..., 49.833333, 49.875   , 49.916667],
      shape=(621,))
Coordinates:
  * lat      (lat) float64 5kB 24.08 24.12 24.17 24.21 ... 49.83 49.88 49.92
Attributes:
    standard_name:  latitude
    long_name:      latitude
    units:          degrees_north